# PopGMM — ancestry-homogeneous sample selection for association analysis

Projects a study cohort onto a population PCA reference panel, models the panel
with a Gaussian mixture, and selects the ancestrally homogeneous subsets of the
cohort that association analysis should run on.

**The deliverable is three sample lists**, written to `results/keep_lists/`:

| List | Definition |
|---|---|
| `full` | every component of the major cluster — the widest defensible set |
| `refined` | the primary analysis set: a rank cut chosen on effective sample size vs residual spread |
| `expanded` | a looser cut between refined and full, for sensitivity analysis |

A fourth list, `reference_full`, holds the reference panel's own major-cluster
samples — not a cohort deliverable, but the same selection applied to the panel.

Run top to bottom; cells form a linear dependency chain and the working
directory must be the repository root. All parameters live in
[`scripts/params.py`](scripts/params.py).

In [1]:
import dataclasses
import json

import matplotlib.pyplot as plt

import scripts.params as params
from scripts.artifacts import ArtifactCache, run_environment

plt.style.use("default")

# "fresh" executes every stage and writes all of its output files. This is the
# default and the only mode valid for publication or verification.
#
# "resume" reuses cached artifacts for the three expensive upstream stages,
# cutting a run from ~13 min to well under a minute while iterating. Two
# measured consequences: those stages write nothing at all, and the FIGURES of
# later stages change -- the data stays byte-identical, but skipping them means
# their plotting code never runs, so later stages inherit a different global
# rcParams state. Never publish figures from a resume run; verify_results.py
# refuses to verify such a tree.
RUN_MODE = "fresh"

cache = ArtifactCache(mode=RUN_MODE)

params.PROVENANCE_DIR.mkdir(parents=True, exist_ok=True)
(params.PROVENANCE_DIR / "run_environment.json").write_text(
    json.dumps(run_environment(RUN_MODE), indent=2, sort_keys=True) + "\n"
)

print(f"results root : {params.RESULTS_ROOT}")
print(f"run mode     : {RUN_MODE}")
print(f"deliverable  : {params.KEEP_LIST_DIR}")

results root : results
run mode     : fresh
deliverable  : results/keep_lists


## Data loading

Reads the shared `.sscore` matrix once and splits it by IID prefix into the
reference panel and the study cohort, deriving case/control lists from the
phenotype column.

In [2]:
from scripts.data_loading import DataLoadingConfig, load_reference_and_study

config_loading = DataLoadingConfig(
    chunksize=50000,
    reference_iid_prefix=params.REFERENCE_IID_PREFIX,
    verbose=True,
    phenotype_column="PHENO1",
    case_value=2,
    control_value=1,
)

eigenval, reference_samples, study_samples, case_iids, control_iids = cache.compute(
    "data_loading",
    lambda: load_reference_and_study(
        eigenval_path=params.EIGENVAL_PATH,
        sscore_path=params.SSCORE_PATH,
        config=config_loading,
    ),
    config=config_loading,
    files=[params.EIGENVAL_PATH, params.SSCORE_PATH],
    writes_side_effects=False,
)

[cache] run  data_loading 625277d64997 (fresh mode)



                 DATA LOADING (REFERENCE PANEL + STUDY COHORT)                  

[CONFIGURATION]
--------------------------------------------------------------------------------
  Eigenvalue path      : data/bbj.pca_base.eigenval
  Score file path      : data/cteph_agp3k_v6_wgs_merged.sample_qc.variant_qc.bbjproj.sscore
  Chunk size           : 50,000
  BBJ prefix           : bbj_
  Phenotype column     : PHENO1
  Case / Control value : 2 / 1

[RESULTS]
--------------------------------------------------------------------------------
  Eigenvalues          : 20 PCs × 4 metrics
  PC1 variance explained: 39.24%
  PC1-2 cumulative var.: 46.70%
  BBJ samples          : 183,013 × 22 cols (49.92 MB)
  OUR samples          : 3,571 × 22 cols (0.94 MB)
  OUR cases / ctrls    : 447 / 3,124



[cache] save data_loading 625277d64997 (0.9s, 28.1 MB)


## Reference panel — denoising

Removes sparse outliers in PC1–PC2 space so the mixture is fitted to stable
population structure rather than to scatter.

`hdbscan_filtering` is pinned to python-hdbscan and errors rather than falling
back to another implementation: the two disagree on the noise set.

In [3]:
from scripts.hdbscan_filtering import HDBSCANConfig, run_hdbscan_denoise

config_denoising = HDBSCANConfig(
    n_pcs_hdbscan=2,
    use_zscale_hdbscan=True,
    min_cluster_size=50,
    min_samples=6,
    cluster_selection_epsilon=0.005,
    cluster_selection_method="eom",
    metric="euclidean",
    alpha=0.8,
    allow_single_cluster=True,
    leaf_size=40,
    algorithm="best",
    approx_min_span_tree=True,
    gen_min_span_tree=False,
    output_dir=params.DENOISING_DIR,
    save_plot=True,
    save_tables=True,
    verbose=True,
)

denoise_out = cache.compute(
    "denoising",
    lambda: run_hdbscan_denoise(
        reference_samples=reference_samples,
        eigenval=eigenval,
        config=config_denoising,
    ),
    config=config_denoising,
    frames=[reference_samples],
)

reference_samples_filtered = denoise_out.reference_samples_filtered.drop(
    columns=["HDBSCAN_Label"], errors="ignore"
)

[cache] run  denoising 230f1091959d (fresh mode)



                      HDBSCAN DENOISING (REFERENCE PANEL)                       

[CONFIGURATION]
--------------------------------------------------------------------------------
  n_pcs_hdbscan         : 2
  min_cluster_size      : 50
  min_samples           : 6
  cluster_epsilon       : 0.005
  cluster_method        : eom
  metric                : euclidean
  alpha                 : 0.8
  allow_single_cluster  : True
  leaf_size             : 40
  algorithm             : best
  approx_min_span_tree  : True
  gen_min_span_tree     : False
  use_zscale            : True
  save_plot             : True
  output_dir            : results/01_reference_model/denoising

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 183,013
  output_rows           : 181,815
  noise_rows            : 1,198
  noise_ratio           : 0.65%
  clusters_found        : 7


[cache] save denoising 230f1091959d (15.6s, 30.5 MB)


## Reference panel — mixture model

Fits full-covariance Gaussian mixtures across the candidate component counts and
selects the minimum-BIC model that has no empty component.

The dominant cost of the pipeline, and the fitted model exists nowhere else,
which is why it is cached. Computation is in float64: under float32 the
log-likelihood sum was sensitive to BLAS reduction order and the search log was
not reproducible between runs.

In [4]:
from scripts.gmm_clustering import GMMConfig, run_gmm_fixed_pcs

config_mixture = GMMConfig(
    fixed_n_pcs=2,
    k_min=2,
    k_max=100,
    use_zscale=False,
    covariance_type="full",
    n_init=3,
    init_params="kmeans",
    reg_covar=1e-6,
    max_iter=200,
    random_state=params.RANDOM_SEED,
    search_max_samples=200000,
    search_workers=6,
    require_non_empty_clusters=True,
    output_dir=params.MIXTURE_DIR,
    save_plot=True,
    save_tables=True,
    verbose=True,
)

mixture_out = cache.compute(
    "mixture_model",
    lambda: run_gmm_fixed_pcs(
        reference_samples_filtered=reference_samples_filtered,
        eigenval=eigenval,
        config=config_mixture,
    ),
    config=config_mixture,
    upstream=["denoising"],
    frames=[reference_samples_filtered],
)

reference_samples_gmm = mixture_out.reference_samples_with_cluster
gmm_summary = mixture_out.summary
gmm_model = mixture_out.model  # needed by every stage below

[cache] run  mixture_model 0c5c94495696 (fresh mode)



                  MIXTURE MODEL (FIXED PCs, MIN-BIC SELECTION)                  

[CONFIGURATION]
--------------------------------------------------------------------------------
  fixed_n_pcs           : 2
  k_range               : 2..100
  covariance_type       : full
  n_init                : 3
  init_params           : kmeans
  reg_covar             : 1e-06
  max_iter              : 200
  random_state          : 42
  search_rows           : 181,815
  full_rows             : 181,815
  search_workers        : 6
  require_non_empty     : True
  use_zscale            : False
  output_dir            : results/01_reference_model/mixture_model

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  best_k                : 26
  best_bic              : -2,682,560.08
  non_empty_models      : 99
  clusters_found        : 26


[cache] save mixture_model 0c5c94495696 (646.0s, 63.4 MB)


## Reference panel — component merging and the major cluster

Merges components by Mahalanobis distance between their means under the pooled
covariance, then cuts the dendrogram at `params.MERGE_THRESHOLD`.

The **major cluster** is the merged cluster holding the most pre-merge
components (ties to the smallest id) — derived, never hard-coded. Its
population-genetic interpretation is an assumption the pipeline does not verify;
`params.MAJOR_CLUSTER_DISPLAY_NAME` is what appears on figures.

In [5]:
from scripts.gmm_component_merging import (
    GMMComponentMergingConfig,
    run_gmm_component_merging,
    summarize_threshold_robustness,
)

config_merging = GMMComponentMergingConfig(
    merge_threshold=params.MERGE_THRESHOLD,
    linkage_method="average",
    output_dir=params.MERGING_DIR,
    save_plot=True,
    save_tables=True,
    # Stable, interpretable legend range across runs for the confidence panel.
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_out = run_gmm_component_merging(
    gmm_model=gmm_model,
    reference_samples_gmm=reference_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_merging,
)

merge_map = merge_out.merge_map
major_cluster_component_ids = merge_out.major_cluster_component_ids
print(f"major cluster: {len(major_cluster_component_ids)} components {major_cluster_component_ids}")


           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 6.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 3
  mainland_premerge_id  : 0
  output_dir            : results/01_reference_model/component_merging

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 6
major cluster: 16 components [0, 2, 4, 5, 6, 8, 10, 11, 12, 13, 16, 18, 19, 20, 21, 22]


## Major cluster — robustness to the merge threshold

Re-runs the merge at the thresholds in `params.MERGE_THRESHOLD_ROBUSTNESS` and
compares which components the major cluster picks up.

This answers whether the identification is stable: a strict subset relationship
means a tighter cut only carves the same region more finely, whereas a low
Jaccard index would mean it jumps elsewhere. `dataclasses.replace` derives each
config from the main one, so every unlisted setting is guaranteed identical.

In [6]:
robustness_results = {params.MERGE_THRESHOLD: merge_out}

for _threshold in params.MERGE_THRESHOLD_ROBUSTNESS:
    _config = dataclasses.replace(
        config_merging,
        merge_threshold=_threshold,
        output_dir=params.threshold_robustness_dir(_threshold),
    )
    robustness_results[_threshold] = run_gmm_component_merging(
        gmm_model=gmm_model,
        reference_samples_gmm=reference_samples_gmm,
        eigenval=eigenval,
        gmm_summary=gmm_summary,
        config=_config,
    )

robustness_table = summarize_threshold_robustness(
    results_by_threshold=robustness_results,
    main_threshold=params.MERGE_THRESHOLD,
    output_path=params.THRESHOLD_ROBUSTNESS_DIR / "major_cluster_robustness.tsv",
)


           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 2.5
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 7
  mainland_premerge_id  : 2
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_2p5

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 12



           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 3.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 6
  mainland_premerge_id  : 2
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_3p0

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 11



           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 3.5
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 3
  mainland_premerge_id  : 0
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_3p5

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 9



           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 4.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 3
  mainland_premerge_id  : 0
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_4p0

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 8



           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 4.5
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 3
  mainland_premerge_id  : 0
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_4p5

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 8



           COMPONENT MERGING (MAHALANOBIS + HIERARCHICAL CLUSTERING)            

[CONFIGURATION]
--------------------------------------------------------------------------------
  linkage_method        : average
  merge_threshold       : 8.0
  covariance_type       : full
  conf_scale_mode       : fixed
  conf_scale_fixed      : 0.950–1.000
  conf_norm             : power
  conf_power_gamma      : 0.400
  conf_scale_hard_floor : None
  save_plot             : True
  save_tables           : True
  mainland_merged_id    : 2
  mainland_premerge_id  : 0
  output_dir            : results/01_reference_model/component_merging/threshold_robustness/threshold_8p0

[RESULTS]
--------------------------------------------------------------------------------
  input_rows            : 181,815
  original_components   : 26
  merged_components     : 5

MAJOR-CLUSTER ROBUSTNESS ACROSS MERGE THRESHOLDS
   threshold  2.5:  12 clusters, major holds   7 components /  94,997 samples (52.25%), subset=True, ja

## Cohort assignment

Projects the study cohort into the mixture and takes `predict_proba` with an
identity label map, so every component stays separate.
`Assignment_Confidence` is the maximum posterior.

The second half compares case and control distributions across all PCs within
the major cluster (Welch *t* + Mann-Whitney, BH-FDR).

In [7]:
from typing import Any, cast

from scripts.cohort_assignment import CohortAssignmentConfig, run_cohort_assignment
from scripts.major_cluster_all_pcs_kde import (
    MajorClusterAllPCsKDEConfig,
    run_major_cluster_all_pcs_kde,
)

# Identity map: each mixture component maps to itself.
n_components = int(getattr(cast(Any, gmm_model), "n_components"))
identity_label_map = {int(k): int(k) for k in range(n_components)}

config_assignment = CohortAssignmentConfig(
    output_dir=params.ASSIGNMENT_DIR,
    save_plot=True,
    save_tables=True,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    reference_alpha=0.20,
    verbose=True,
)

assignment_out = run_cohort_assignment(
    gmm_model=gmm_model,
    reference_samples_gmm=reference_samples_gmm,
    study_samples=study_samples,
    case_iids=case_iids,
    control_iids=control_iids,
    label_map=identity_label_map,
    merge_map=merge_map,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    training_use_zscale=config_mixture.use_zscale,
    config=config_assignment,
)

config_major_kde = MajorClusterAllPCsKDEConfig(
    output_dir=params.ASSIGNMENT_DIR,
    save_plot=True,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    reference_color="#1F78B4",
    case_color="#E31A1C",
    alpha=0.65,
    verbose=True,
)

major_kde_out = run_major_cluster_all_pcs_kde(
    df_results=assignment_out.df_results,
    study_samples=study_samples,
    case_iids=case_iids,
    control_iids=control_iids,
    major_cluster_component_ids=major_cluster_component_ids,
    eigenval=eigenval,
    config=config_major_kde,
)


                 COHORT ASSIGNMENT TO PRE-MERGE GMM COMPONENTS                  

[CONFIGURATION]
--------------------------------------------------------------------------------
  output_dir            : results/02_cohort_assignment
  save_tables           : True
  save_plot             : True
  show_plot             : False

[RESULTS]
--------------------------------------------------------------------------------
  cohort rows           : 3,571
  assigned_clusters (K) : 26
  assignment_tsv        : results/02_cohort_assignment/cohort_posterior_probabilities.tsv
  mainland_cluster_rank : results/02_cohort_assignment/major_cluster_component_ranks.tsv
>>> ANALYZING ALL PC DISTRIBUTIONS FOR MAINLAND SAMPLES...


   -> output_dir = results/02_cohort_assignment


   -> mainland clusters = [0, 2, 4, 5, 6, 8, 10, 11, 12, 13, 16, 18, 19, 20, 21, 22]


   -> Mainland samples: 3099 / 3571


   -> Case samples (Mainland): 434


...

   -> Running statistical tests...


   -> Applying FDR correction (Benjamini-Hochberg method) to all tests...


   -> FDR correction complete.


      • Significant by t-test: 3 PC(s)


      • Significant by Mann-Whitney U: 4 PC(s)


## Rank selection — effective sample size vs residual spread

Ranks the major cluster's components by case/control ratio, then walks the
cumulative sets: including the top-k trades **GWAS_Neff** (effective sample
size, `4 / (1/n_case + 1/n_control)`) against **PC12_RGV** (residual genetic
spread, `det(Sigma)**0.25` on PC1–PC2). Reports the Pareto front.

This stage produces the *evidence*; the cut itself is a human decision recorded
in `params.REFINED_RANK_K` and `params.EXPANDED_RANK_K`. Set either to
`"pareto"` to delegate it to the Pareto optimum instead.

In [8]:
from scripts.rank_selection import RankSelectionConfig, run_rank_selection

config_rank = RankSelectionConfig(
    output_dir=params.RANK_SELECTION_DIR,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    forced_recommended_rank=params.REFINED_RANK_K,  # None -> Pareto optimum
    save_plot=True,
    show_plot=False,
    verbose=True,
)

rank_out = run_rank_selection(
    df_results=assignment_out.df_results,
    merge_map=merge_map,
    case_iids=case_iids,
    control_iids=control_iids,
    gmm_model=gmm_model,
    gmm_summary=gmm_summary,
    config=config_rank,
)

rank_table = rank_out.rank_table
print(f"Pareto/forced recommended rank: {rank_out.recommended_rank}")


                  RANK SELECTION: EFFECTIVE SAMPLE SIZE vs RESIDUAL SPREAD                  
Mainland clusters ranked (top 16): [13, 5, 12, 0, 4, 19, 2, 16, 22, 18, 20, 8, 10, 11, 21, 6]
Recommended rank k   : 10  (forced)
Rank table saved      : results/03_rank_selection/component_rank_table.tsv
Cumulative table saved: results/03_rank_selection/rank_cumulative_metrics.tsv
Decision table saved  : results/03_rank_selection/rank_decision_table.tsv
Figure saved          : results/03_rank_selection/rank_selection_tradeoff.png
--------------------------------------------------------------------------------------------
 Included_Max_Rank  Included_Cluster_Count                         Included_Clusters  CTEPH_Count  AGP3K_Count  Case_Control_Ratio  Total_Count   GWAS_Neff  PC12_AllSample_Heterogeneity  Delta_Neff  Delta_Heterogeneity  Neff_Gain_per_Heterogeneity  Neff_Norm  Heterogeneity_Norm  Utility_NeffMinusHet  Is_Pareto  Distance_To_Ideal  Is_Recommended
                 1             

## Subcluster variants

For each variant, merges the selected major-cluster components into one
composite group, renormalizes the posteriors over the resulting groups and
reassigns by argmax. Every other component stays separate, so a borderline
sample is absorbed only when its joint subcluster posterior beats every single
outside component.

`full` runs the same way with nothing excluded, which makes all three variants
directly comparable: each gets its own directory under
`04_subcluster_variants/` with the posterior table, the PC1–PC2 view and the
all-PC KDE panel, produced by identical code.

In [9]:
from scripts.subcluster_assignment import SubclusterAssignmentConfig, run_subcluster_assignment
from scripts.subcluster_view import SubclusterViewConfig, run_subcluster_view
from scripts.subcluster_all_pcs_kde import SubclusterAllPCsKDEConfig, run_subcluster_all_pcs_kde

GROUP_LABEL = f"{params.MAJOR_CLUSTER_DISPLAY_NAME} Subcluster"
ASSIGNED_GROUP_COL = "Assigned_Mainland_Subcluster_Group"

variant_results: dict[str, dict] = {}

for _variant, _cut in params.SUBCLUSTER_VARIANTS.items():
    # "full" keeps every major-cluster component, so nothing is excluded and the
    # variant carries no rank. Any other cut keeps the components ranked 1..k.
    if _cut == "full":
        _rank = None
        _excluded = ()
    else:
        _resolved = rank_out.recommended_rank if _cut == "pareto" else _cut
        if _resolved is None:
            raise ValueError(
                f"variant {_variant!r} asks for the Pareto rank, but the rank-selection "
                f"analysis produced no recommendation. Set an explicit rank in "
                f"params.SUBCLUSTER_VARIANTS."
            )
        _rank = int(_resolved)
        _included = {int(v) for v in rank_table.loc[rank_table["Rank"] <= _rank, "Cluster"]}
        _excluded = tuple(sorted({int(v) for v in major_cluster_component_ids} - _included))

    _dir = params.subcluster_dir(_variant)

    _assign = run_subcluster_assignment(
        gmm_model=gmm_model,
        reference_samples_gmm=reference_samples_gmm,
        study_samples=study_samples,
        case_iids=case_iids,
        control_iids=control_iids,
        major_cluster_component_ids=major_cluster_component_ids,
        eigenval=eigenval,
        gmm_summary=gmm_summary,
        config=SubclusterAssignmentConfig(
            output_dir=_dir,
            save_plot=True,
            save_tables=True,
            group_label=GROUP_LABEL,
            exclude_cluster_ids=_excluded,
            case_label=params.CASE_LABEL,
            control_label=params.CONTROL_LABEL,
            reference_alpha=config_assignment.reference_alpha,
            verbose=True,
        ),
    )

    _view = run_subcluster_view(
        df_assigned=_assign.df_results,
        case_iids=case_iids,
        control_iids=control_iids,
        reference_samples_gmm=reference_samples_gmm,
        eigenval=eigenval,
        config=SubclusterViewConfig(
            output_dir=_dir,
            group_label=GROUP_LABEL,
            assigned_group_col=ASSIGNED_GROUP_COL,
            case_label=params.CASE_LABEL,
            control_label=params.CONTROL_LABEL,
            save_plot=True,
            show_plot=False,
            verbose=True,
        ),
    )

    _kde = run_subcluster_all_pcs_kde(
        df_assigned=_assign.df_results,
        study_samples=study_samples,
        case_iids=case_iids,
        control_iids=control_iids,
        config=SubclusterAllPCsKDEConfig(
            output_dir=_dir,
            group_label=GROUP_LABEL,
            assigned_group_col=ASSIGNED_GROUP_COL,
            case_label=params.CASE_LABEL,
            control_label=params.CONTROL_LABEL,
            reference_color="#1F78B4",
            case_color="#E31A1C",
            alpha=0.65,
            verbose=True,
        ),
    )

    variant_results[_variant] = {
        "rank": _rank,
        "components": _assign.subcluster_components,
        "frame": _kde.df_subcluster,
    }
    print(f"[{_variant}] rank {'none (full)' if _rank is None else _rank}: "
          f"{len(_assign.subcluster_components)} components, "
          f"{len(_kde.df_subcluster):,} samples")

# With nothing excluded the composite group is the whole merged cluster, so the
# uncut variant should reproduce the major cluster exactly as the cohort
# assignment stage defined it. Reported rather than asserted, so a divergence
# shows up in the log and in the keep-list comparison instead of killing the run.
_full_iids = set(variant_results["full"]["frame"]["IID"].astype(str))
_major_iids = set(major_kde_out.df_major_cluster["IID"].astype(str))
print(
    f"\nfull variant vs cohort-assignment major cluster: "
    f"{len(_full_iids):,} vs {len(_major_iids):,} samples, "
    + ("identical" if _full_iids == _major_iids else f"DIFFER on {len(_full_iids ^ _major_iids)}")
)


                            SUBCLUSTER REASSIGNMENT                             
  mainland_subcluster_ids: [0, 2, 4, 5, 6, 8, 10, 11, 12, 13, 16, 18, 19, 20, 21, 22]
  remaining_component_ids: [1, 3, 7, 9, 14, 15, 17, 23, 24, 25]
  excluded_cluster_ids   : []
  output_dir             : results/04_subcluster_variants/full
  assignment_tsv         : results/04_subcluster_variants/full/subcluster_posterior_probabilities.tsv



                                SUBCLUSTER VIEW                                 
  assigned_group_col : Assigned_Mainland_Subcluster_Group
  mainland_label     : Mainland Subcluster
  rows (unfiltered)  : 3099
  unlabeled rows     : 0
  figure_file        : results/04_subcluster_variants/full/subcluster_view.png
>>> ANALYZING ALL PC DISTRIBUTIONS FOR MAINLAND_SUBCLUSTER SAMPLES...


   -> output_dir = results/04_subcluster_variants/full


   -> mainland label = Mainland Subcluster


   -> mainland_subcluster samples = 3099 / 3571


...

   -> Applying FDR correction (Benjamini-Hochberg method) to all tests...


   -> FDR correction complete.


      • Significant by t-test: 5 PC(s)


      • Significant by Mann-Whitney U: 5 PC(s)


[expanded] rank 12: 12 components, 2,637 samples

full variant vs cohort-assignment major cluster: 3,099 vs 3,099 samples, identical


## Keep lists — the deliverable

Writes the three cohort sample lists and the table comparing them, plus the
reference panel's own major cluster. A list is a headerless tab-separated
`FID IID` file:

```bash
plink2 --pfile <dataset> \
       --keep results/keep_lists/refined_mainland.fid_iid.txt \
       --make-pgen --out <dataset>.ancestry_qc
```

`reference_full_mainland.fid_iid.txt` holds the BBJ samples in the same major
cluster. Its case and control counts are zero because the reference panel is not
part of the case/control cohort; its `PC12_RGV` is the useful number — the
residual spread of the reference region the cohort variants are approximating.

In [10]:
from scripts.keep_lists import KeepListConfig, KeepListVariant, write_keep_lists

# The reference panel's own major cluster, for anyone who needs the BBJ side of
# the same selection (e.g. to re-derive the PCA, or as an ancestry-matched
# external control set). Selected on the pre-merge component, exactly as the
# cohort variants are.
reference_major_cluster = reference_samples_gmm.loc[
    reference_samples_gmm["GMM_Cluster"].isin(list(major_cluster_component_ids))
].copy()

keep_list_out = write_keep_lists(
    variants=[
        *[
            KeepListVariant(
                name=name,
                frame=res["frame"],
                rank_cut=res["rank"],
                component_ids=res["components"],
            )
            for name, res in variant_results.items()
        ],
        KeepListVariant(
            name="reference_full",
            frame=reference_major_cluster,
            rank_cut=None,
            component_ids=major_cluster_component_ids,
        ),
    ],
    case_iids=case_iids,
    control_iids=control_iids,
    config=KeepListConfig(
        output_dir=params.KEEP_LIST_DIR,
        name_suffix=params.MAJOR_CLUSTER_DISPLAY_NAME.lower(),
        case_label=params.CASE_LABEL,
        control_label=params.CONTROL_LABEL,
        verbose=True,
    ),
)

keep_list_out.summary


KEEP LISTS (deliverable)
  full      n= 3,099  CTEPH= 434  AGP3K=2,665  Neff=  1492.88  RGV=0.006962
  refined   n= 2,417  CTEPH= 424  AGP3K=1,993  Neff=  1398.48  RGV=0.005137
  expanded  n= 2,637  CTEPH= 430  AGP3K=2,207  Neff=  1439.53  RGV=0.005864
  reference_full n=167,694  CTEPH=   0  AGP3K=    0  Neff=      nan  RGV=0.005569
  -> results/keep_lists


,variant,rank_cut,n_components,n_samples,CTEPH_Count,AGP3K_Count,Case_Control_Ratio,GWAS_Neff,PC12_RGV,components,file
0,full,,16,3099,434,2665,0.162852,1492.881575,0.006962,"0,2,4,5,6,8,10,11,12,13,16,18,19,20,21,22",full_mainland.fid_iid.txt
1,refined,10,10,2417,424,1993,0.212745,1398.480761,0.005137,"0,2,4,5,12,13,16,18,19,22",refined_mainland.fid_iid.txt
2,expanded,12,12,2637,430,2207,0.194835,1439.529769,0.005864,"0,2,4,5,8,12,13,16,18,19,20,22",expanded_mainland.fid_iid.txt
3,reference_full,,16,167694,0,0,NaN,NaN,0.005569,"0,2,4,5,6,8,10,11,12,13,16,18,19,20,21,22",reference_full_mainland.fid_iid.txt


## Provenance

Serializes every stage config. Diffing two snapshots proves a refactor did not
alter a parameter *without* re-running the pipeline, which makes it the cheap
pre-flight check before spending a full run on verification.

In [11]:
_configs = {
    "loading": config_loading,
    "denoising": config_denoising,
    "mixture_model": config_mixture,
    "component_merging": config_merging,
    "cohort_assignment": config_assignment,
    "major_cluster_kde": config_major_kde,
    "rank_selection": config_rank,
}

_snapshot = {name: dataclasses.asdict(cfg) for name, cfg in _configs.items()}
_snapshot["_derived"] = {
    "major_cluster_component_ids": [int(v) for v in major_cluster_component_ids],
    "recommended_rank": rank_out.recommended_rank,
    "subcluster_variants": {
        name: {"rank": res["rank"], "components": [int(c) for c in res["components"]]}
        for name, res in variant_results.items()
    },
    "merge_threshold_robustness": [float(t) for t in params.MERGE_THRESHOLD_ROBUSTNESS],
}

_path = params.PROVENANCE_DIR / "run_config_snapshot.json"
_path.write_text(json.dumps(_snapshot, indent=2, sort_keys=True, default=str) + "\n")
print(f"wrote {len(_configs)} stage configs -> {_path}")

wrote 7 stage configs -> results/provenance/run_config_snapshot.json
